# RIBBIT Pulse Rate model demonstration
RIBBIT (Repeat-Interval Based Bioacoustic Identification Tool) is a tool for detecting vocalizations that have a repeating structure.

This tool is useful for detecting vocalizations of frogs, toads, and other animals that produce vocalizations with a periodic structure. In this notebook, we demonstrate how to select model parameters for the Great Plains Toad, then run the model on data to detect vocalizations.

This work is described in:

* 2021 paper, ["Automated detection of frog calls and choruses by pulse repetition rate"](https://doi.org/10.1111/cobi.13718)

* 2020 poster, ["Automatic Detection of Pulsed Vocalizations"](https://f1000research.com/posters/9-964)

RIBBIT is also [available as an R package](https://github.com/kitzeslab/r-ribbit).

This notebook demonstrates how to use the RIBBIT tool implemented in opensoundscape as `opensoundscape.ribbit.ribbit()`

For help instaling OpenSoundscape, see the [documentation](https://opensoundscape.org)

## Import packages

In [ ]:
# import packages
import numpy as np
from glob import glob
import pandas as pd

# local imports from opensoundscape
from opensoundscape.audio import Audio
from opensoundscape.spectrogram import Spectrogram
from opensoundscape.ribbit import ribbit

### Download example audio
First, let's download some example audio to work with.

Download the zip file from Google Drive, then un-zip the folder. Specify the path to the folder in the cell below

https://drive.google.com/file/d/1oBzif4Gez3ZRYLhiwRg6L5hpHyccXG51/view?usp=sharing


In [ ]:
dataset_path = "./great_plains_toad_dataset/"

# let's make sure the files are where we expect them:
mp3s_found = len(glob(f"{dataset_path}/*.mp3"))
print(f"Found {mp3s_found} mp3 files in {dataset_path}, expected 18")

Found 18 mp3 files in ./great_plains_toad_dataset/, expected 18


## Select model parameters

RIBBIT requires the user to select a set of parameters that describe the target vocalization. Here is some detailed advice on how to use these parameters.

**Signal Band:** The signal band is the frequency range where RIBBIT looks for the target species. Based on the spectrogram above, we can see that the Great Plains Toad vocalization has the strongest energy around 2000-2500 Hz, so we will specify `signal_band = [2000,2500]`. It is best to pick a narrow signal band if possible, so that the model focuses on a specific part of the spectrogram and has less potential to include erronious sounds. 

**Noise Bands:** Optionally, users can specify other frequency ranges called noise bands. Sounds in the `noise_bands` are _subtracted_ from the `signal_band`. Noise bands help the model filter out erronious sounds from the recordings, which could include confusion species, background noise, and popping/clicking of the microphone due to rain, wind, or digital errors. It's usually good to include one noise band for very low frequencies -- this specifically eliminates popping and clicking from being registered as a vocalization. It's also good to specify noise bands that target confusion species. Another approach is to specify two narrow `noise_bands` that are directly above and below the `signal_band`. 

**Pulse Rate Range:** This parameters specifies the minimum and maximum pulse rate (the number of pulses per second, also known as pulse repetition rate) RIBBIT should look for to find the focal species. Looking at the spectrogram above, we can see that the pulse rate of this Great Plains Toad vocalization is about 15 pulses per second. By looking at other vocalizations in different environmental conditions, we notice that the pulse rate can be as slow as 10 pulses per second or as fast as 20. So, we choose `pulse_rate_range = [10, 20]` meaning that RIBBIT should look for pulses no slower than 10 pulses per second and no faster than 20 pulses per second. 

**Clip Duration:** This parameter tells the algorithm how many seconds of audio to analyze at one time. Generally, you should choose a `clip_duration` that is ~2x longer than the target species vocalization, or a little bit longer. For very slowly pulsing vocalizations, choose a longer window so that at least 5 pulses can occur in one window (0.5 pulses per second -> 10 second window). Typical values for `clip_duration` are 0.3 to 10 seconds. Here, because the The Great Plains Toad has a vocalization that continues on for many seconds (or minutes!), we chose a 2-second window which will include plenty of pulses. 

- we can also set `clip_overlap` if we want overlapping clips. For instance, a `clip_duration` of 2 with `clip_overlap` of 1 results in 50% overlap of each consecutive clip. This can help avoid sounds being split up across two clips, and therefore not being detected. 
- `final_clip` determines what should be done when there is less than `clip_duration` audio remaining at the end of an audio file. We'll just use `final_clip=None` to discard any remaining audio that doesn't make a complete clip. 

**Plot:** We can choose to show the power spectrum of pulse repetition rate for each window by setting `plot=True`. The default is not to show these plots (`plot=False`).

In [6]:
# minimum and maximum rate of pulsing (pulses per second) to search for
pulse_rate_range = [8, 15]

# look for a vocalization in the range of 1000-2000 Hz
signal_band = [1800, 2400]

# subtract the amplitude signal from these frequency ranges
noise_bands = [[0, 1000], [3000, 3200]]

# divides the signal into segments this many seconds long, analyzes each independently
clip_duration = 2  # seconds
clip_overlap = 0  # seconds

# if True, it will show the power spectrum plot for each audio segment
show_plots = True

## Search for pulsing vocalizations with `ribbit()` 

This function takes the parameters we chose above as arguments, performs the analysis, and returns two arrays:
 - **scores:** the pulse rate score for each window
 - **times:** the start time in seconds of each window
 
The scores output by the function may be very low or very high. They do not represent a "confidence" or "probability" from 0 to 1. Instead, the relative values of scores on a set of files should be considered: when RIBBIT detects the target species, the scores will be significantly higher than when the species is not detected. 

## Analyzing a set of files

In [ ]:
# set up a dataframe for storing files' scores and labels
df = pd.DataFrame(
    index=glob("./great_plains_toad_dataset/*.mp3"), columns=["score", "label"]
)

# label is 1 if the file contains a Great Plains Toad vocalization, and 0 if it does not
df["label"] = [1 if "gpt" in f else 0 for f in df.index]

# calculate RIBBIT scores
for path in df.index:

    # make the spectrogram
    spec = Spectrogram.from_audio(Audio.from_file(path))

    # run RIBBIT
    score_df = ribbit(
        spec,
        pulse_rate_range=[8, 20],
        signal_band=[1900, 2400],
        clip_duration=clip_duration,
        noise_bands=[[0, 1500], [2500, 3500]],
        plot=False,
    )

    # optionaly threshold and save sparse score df

    # save score df for the file

Files sorted by score, from highest to lowest:


,file,score,label,start_time
4,./great_plains_toad_dataset/gpt0.mp3,37.97,1,0
6,./great_plains_toad_dataset/gpt2.mp3,11.36,1,0
7,./great_plains_toad_dataset/gpt3.mp3,11.26,1,0
5,./great_plains_toad_dataset/negative9.mp3,10.65,0,0
3,./great_plains_toad_dataset/gpt1.mp3,4.67,1,0
16,./great_plains_toad_dataset/negative2.mp3,1.66,0,0
11,./great_plains_toad_dataset/negative4.mp3,1.37,0,0
2,./great_plains_toad_dataset/pops2.mp3,1.06,0,0
8,./great_plains_toad_dataset/negative8.mp3,0.85,0,0
0,./great_plains_toad_dataset/pops1.mp3,0.76,0,0
